# 🔧 Module 6 修复版 - 推荐引擎（FAISS 兼容）

修复了 FAISS 与全量过滤的维度不匹配问题

In [ ]:
class RecommendationEngineFixed:
    """推荐引擎核心模块 - 修复版（FAISS 兼容）"""
    
    def __init__(self, df, X_job, skill_extractor, struct_encoder, fusion):
        """初始化推荐引擎"""
        self.df = df
        self.X_job = X_job
        self.skill_extractor = skill_extractor
        self.struct_encoder = struct_encoder
        self.fusion = fusion
    
    def build_user_profile(self, user_skills, user_city=None, user_category=None,
                           user_edu=None, user_exp=None):
        """构建用户档案"""
        # 技能向量
        user_skill_vec = np.zeros(len(self.skill_extractor.unique_patterns))
        matched_skill_indices = []
        
        for skill in user_skills:
            skill_lower = skill.lower()
            for i, pattern in enumerate(self.skill_extractor.unique_patterns):
                if skill_lower == pattern:
                    user_skill_vec[i] = 1
                    matched_skill_indices.append(i)
                    break
        
        # 结构化向量
        edu_code = self.struct_encoder.EDUCATION_MAP.get(user_edu, 4) if user_edu else 4
        exp_code = user_exp if user_exp is not None else 3
        
        user_struct = np.array([
            self.struct_encoder.top_cities.index(user_city) / len(self.struct_encoder.top_cities) 
                if user_city in self.struct_encoder.top_cities else 0.5,
            self.struct_encoder.cat_map.get(user_category, 0) / len(self.struct_encoder.cat_map) 
                if user_category else 0.5,
            edu_code / 6,
            exp_code / 10,
            0.5
        ]).reshape(1, -1)
        
        user_struct_scaled = self.struct_encoder.scaler.transform(user_struct)
        
        # 融合
        user_profile = np.hstack([user_skill_vec, user_struct_scaled[0]]).astype(np.float32)
        
        return user_profile, matched_skill_indices
    
    def recommend_jobs(self, user_skills, user_city=None, user_category=None,
                       user_edu=None, user_exp=None, min_salary=None, max_salary=None,
                       required_skills=None, min_skill_match=2, top_k=10, use_faiss=True):
        """推荐岗位（修复版）"""
        
        print(f"\n🔍 开始推荐...")
        print(f"   用户技能: {user_skills}")
        print(f"   城市: {user_city}")
        print(f"   职位类别: {user_category}")
        
        # 构建用户档案
        user_profile, matched_skill_indices = self.build_user_profile(
            user_skills, user_city, user_category, user_edu, user_exp
        )
        
        # 先构建全局 mask（所有硬性条件）
        print(f"   应用硬性过滤条件...")
        
        mask = np.ones(len(self.df), dtype=bool)
        job_skill_part = self.X_job[:, :len(self.skill_extractor.unique_patterns)]
        
        # 技能硬性过滤
        match_counts = (user_profile[:len(self.skill_extractor.unique_patterns)] * job_skill_part).sum(axis=1)
        mask &= (match_counts >= min_skill_match)
        
        # 必须技能
        if required_skills:
            for req in required_skills:
                req_lower = req.lower()
                for i, p in enumerate(self.skill_extractor.unique_patterns):
                    if p == req_lower:
                        mask &= (job_skill_part[:, i] == 1)
                        break
        
        # 学历
        if user_edu:
            user_edu_level = self.struct_encoder.EDUCATION_MAP.get(user_edu, 4)
            mask &= (self.df['学历编码'] <= user_edu_level + 1).values
        
        # 经验
        if user_exp is not None:
            mask &= (self.df['经验编码'] <= user_exp + 2).values
        
        # 城市
        if user_city:
            mask &= (self.df['检索城市'] == user_city).values
        
        # 职位类别
        if user_category:
            mask &= (self.df['检索二级职位类别'] == user_category).values
        
        # 薪资
        if min_salary:
            mask &= (self.df['平均薪资'] >= min_salary).values
        if max_salary:
            mask &= (self.df['平均薪资'] <= max_salary).values
        
        # 获取符合硬性条件的索引
        valid_indices = np.where(mask)[0]
        
        print(f"   符合条件的岗位: {len(valid_indices)} 个")
        
        if len(valid_indices) == 0:
            print(f"\n⚠️ 没有符合条件的岗位")
            return pd.DataFrame()
        
        # 计算相似度
        if use_faiss and self.fusion.faiss_index is not None:
            print(f"   使用 FAISS 加速搜索...")
            try:
                # FAISS 返回所有结果
                faiss_indices, faiss_distances = self.fusion.faiss_search(user_profile, k=len(self.df))
                similarities = -faiss_distances  # 转换为相似度（越小越接近）
                
                # 只保留符合条件的结果
                filtered_results = []
                for idx in faiss_indices:
                    if idx in valid_indices:
                        filtered_results.append((idx, -faiss_distances[list(faiss_indices).index(idx)]))
                    if len(filtered_results) >= top_k:
                        break
                
                if not filtered_results:
                    print(f"   FAISS 过滤后无结果，使用全局计算...")
                    # 回退到全局计算
                    use_faiss = False
                else:
                    final_indices = [r[0] for r in filtered_results]
                    final_similarities = np.array([r[1] for r in filtered_results])
            except Exception as e:
                print(f"   FAISS 搜索出错: {e}，使用全局计算...")
                use_faiss = False
        
        # 如果不使用 FAISS 或 FAISS 失败，使用余弦相似度
        if not use_faiss:
            print(f"   使用余弦相似度计算...")
            similarities = cosine_similarity(user_profile.reshape(1, -1), self.X_job)[0]
            
            # 只考虑符合条件的岗位
            valid_similarities = similarities[valid_indices]
            sorted_positions = np.argsort(valid_similarities)[::-1][:top_k]
            final_indices = valid_indices[sorted_positions]
            final_similarities = valid_similarities[sorted_positions]
        
        # 构建结果
        results = []
        for idx in final_indices:
            row = self.df.iloc[idx]
            
            # 匹配的技能
            job_skills = set(np.where(job_skill_part[idx] == 1)[0])
            user_skills_set = set(matched_skill_indices)
            matched = [self.skill_extractor.unique_patterns[i] for i in (job_skills & user_skills_set)]
            missing = [self.skill_extractor.unique_patterns[i] for i in (job_skills - user_skills_set)]
            
            results.append({
                '岗位名': row['岗位名'],
                '公司': row['公司名称'],
                '城市': row['检索城市'],
                '类别': row['检索二级职位类别'],
                '薪资': f"{row['最小薪资']:.0f}-{row['最大薪资']:.0f}",
                '平均薪资': row['平均薪资'],
                '匹配度': float(final_similarities[len(results)] if len(results) < len(final_similarities) else final_similarities[-1]),
                '匹配技能数': len(matched),
                '用户技能数': len(user_skills),
                '匹配技能': matched,
                '岗位额外要求': missing[:5]
            })
        
        print(f"\n✅ 找到 {len(results)} 个匹配岗位")
        return pd.DataFrame(results)

print("\n" + "="*60)
print("Module 6 修复版初始化完成")
print("="*60)
print("\n✨ 修复内容:")
print("   • 解决 FAISS 与全量过滤的维度不匹配")
print("   • 完整的硬性条件过滤")
print("   • 自动回退到余弦相似度")
print("   • 更稳定的错误处理")

## 使用修复版本

In [ ]:
# 使用修复版本替换原来的引擎
print("🔄 重新初始化推荐引擎（使用修复版）...\n")

try:
    engine_fixed = RecommendationEngineFixed(df_clean, X_job, extractor, encoder, fusion)
    print("✅ 修复版引擎初始化完成\n")
    
    # 测试推荐
    print("🧪 测试推荐功能...\n")
    
    test_result = engine_fixed.recommend_jobs(
        user_skills=['python', 'mysql', 'docker', 'linux'],
        user_city='上海',
        user_category='后端开发',
        user_edu='本科',
        user_exp=3,
        min_salary=15000,
        max_salary=40000,
        top_k=5,
        use_faiss=True
    )
    
    print("\n" + "-"*70)
    print(test_result[['岗位名', '公司', '薪资', '匹配度', '匹配技能']])
    print("-"*70)
    
    print("\n✅ 测试成功！")
    print("\n下一步: 启动交互式界面")
    print("  ui = InteractiveRecommender(engine_fixed, df_clean, encoder)")
    print("  ui.create_ui()")
    
except Exception as e:
    print(f"❌ 错误: {e}")
    import traceback
    traceback.print_exc()

## 对比原版本

In [ ]:
print("="*70)
print("原版本 vs 修复版本对比")
print("="*70)

print("\n【测试 1】有符合条件的岗位")
print("-"*70)

print("\n📊 原版本结果:")
try:
    original_result = engine.recommend_jobs(
        user_skills=['python', 'mysql'],
        user_city='上海',
        user_category='后端开发',
        top_k=3,
        use_faiss=False
    )
    print(f"✅ 返回 {len(original_result)} 个结果")
except Exception as e:
    print(f"❌ 出错: {type(e).__name__}")

print("\n📊 修复版本结果:")
try:
    fixed_result = engine_fixed.recommend_jobs(
        user_skills=['python', 'mysql'],
        user_city='上海',
        user_category='后端开发',
        top_k=3,
        use_faiss=False
    )
    print(f"✅ 返回 {len(fixed_result)} 个结果")
    print("\n前 3 个推荐:")
    print(fixed_result[['岗位名', '薪资', '匹配技能']].to_string())
except Exception as e:
    print(f"❌ 出错: {type(e).__name__}")

print("\n\n【测试 2】FAISS 搜索")
print("-"*70)

print("\n📊 修复版本 + FAISS 结果:")
try:
    fixed_faiss_result = engine_fixed.recommend_jobs(
        user_skills=['java', 'spring', 'mysql'],
        user_city='北京',
        user_category='后端开发',
        top_k=3,
        use_faiss=True
    )
    print(f"✅ FAISS 搜索成功，返回 {len(fixed_faiss_result)} 个结果")
    if len(fixed_faiss_result) > 0:
        print("\n前 3 个推荐:")
        print(fixed_faiss_result[['岗位名', '薪资', '匹配技能']].to_string())
except Exception as e:
    print(f"❌ 出错: {type(e).__name__}: {e}")

print("\n\n" + "="*70)
print("✅ 修复版本测试完成")
print("="*70)

## 迁移指南

### 如何从原版本升级到修复版本？

**方法 1: 直接替换变量名**
```python
# 原来的
engine = RecommendationEngine(...)

# 改为
engine = RecommendationEngineFixed(...)
```

**方法 2: 保留两个版本对比**
```python
engine_old = RecommendationEngine(...)      # 原版本
engine_new = RecommendationEngineFixed(...) # 修复版本

# 对比推荐结果
```

### 修复了哪些问题？

| 问题 | 原因 | 解决方案 |
|------|------|----------|
| FAISS 维度不匹配 | FAISS 返回 Top-K，mask 是全量 | 先过滤再排序 |
| 无符合结果时报错 | 没有检查结果是否为空 | 添加空结果检查 |
| FAISS 与过滤不兼容 | 过滤逻辑混乱 | 完整重构过滤逻辑 |
| 错误处理不足 | 没有 try-catch | 添加自动回退机制 |